In [0]:
# =====================================================
# JOB PARAMETERS
# Lakeflow Jobs task parameters (or existing interactive widgets) take precedence;
# a widget is created with its default only when it does not exist yet.
# =====================================================
_PARAM_DEFAULTS = {
    "run_profiling": "false",
    "refresh_reference": "false",
    "strict_expected_counts": "true",
}


def _read_flag(name, default):
    """Read an existing widget / job parameter first; create the widget only if it is missing."""
    try:
        raw, origin = dbutils.widgets.get(name), "existing widget / job parameter"
    except NameError:                               # not running on Databricks
        raw, origin = default, "default (no dbutils)"
    except Exception:                               # widget not defined yet -> create it once
        dbutils.widgets.text(name, default)
        raw, origin = default, "default (widget created)"
    value = str(raw).strip().lower()
    if value not in ("true", "false"):
        raise ValueError(f"Parameter {name}={raw!r} must be 'true' or 'false'")
    return value == "true", origin


_params = {name: _read_flag(name, default) for name, default in _PARAM_DEFAULTS.items()}
RUN_PROFILING = _params["run_profiling"][0]
REFRESH_REFERENCE = _params["refresh_reference"][0]
STRICT_EXPECTED_COUNTS = _params["strict_expected_counts"][0]

for name, (value, origin) in _params.items():
    note = "" if str(value).lower() == _PARAM_DEFAULTS[name] else "  <-- differs from default, confirm this is intended"
    print(f"{name:24s} = {str(value).lower():5s} ({origin}){note}")

if REFRESH_REFERENCE:
    raise RuntimeError("refresh_reference=true is disabled until the airline/airport naming mismatch is resolved. "
                       "No reference table was read or written.")

In [0]:
if not REFRESH_REFERENCE:
    print("Reference refresh skipped (refresh_reference=false) - no JDBC read")
if REFRESH_REFERENCE:
    jdbc_url = (
        "jdbc:sqlserver://capstone3server.database.windows.net:1433;"
        "database=capstone3;"
        "encrypt=true;"
        "trustServerCertificate=false;"
        "hostNameInCertificate=*.database.windows.net;"
        "loginTimeout=30;"
    )
    
    try:
        df = (spark.read
            .format("jdbc")
            .option("url", jdbc_url)
            .option("dbtable", "dbo.airline_id")
            .option("user", "ltmadmin")
            .option("password", "P@55word")
            .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
            .load())
        display(df)
    except Exception as e:
        print(str(e)[:2000])

In [0]:
if not REFRESH_REFERENCE:
    print("Reference refresh skipped (refresh_reference=false) - no JDBC read")
if REFRESH_REFERENCE:
    jdbc_url = (
        "jdbc:sqlserver://capstone3server.database.windows.net:1433;"
        "database=capstone3;"
        "encrypt=true;"
        "trustServerCertificate=false;"
        "hostNameInCertificate=*.database.windows.net;"
        "loginTimeout=30;"
    )
    
    try:
        df1 = (spark.read
            .format("jdbc")
            .option("url", jdbc_url)
            .option("dbtable", "dbo.airport_id")
            .option("user", "ltmadmin")
            .option("password", "P@55word")
            .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
            .load())
        display(df1)
    except Exception as e:
        print(str(e)[:2000])

In [0]:
# DISABLED: this legacy block writes airline_id / airport_id, not the validated airline / airport tables.
# It is unreachable while refresh_reference is false (the parameter cell also stops the run if it is true).
if not REFRESH_REFERENCE:
    print("Reference write skipped (refresh_reference=false) - no reference table written")
if REFRESH_REFERENCE:
    # Save airline data
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("capstone_3flight.flight_schema.airline_id")
    
    
    # Save airport data
    df1.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("capstone_3flight.flight_schema.airport_id")
    
    print("Both tables successfully saved as Delta tables!")

In [0]:
# Read saved Delta tables
saved_airline_df = spark.table(
    "capstone_3flight.flight_schema.airline"
)

saved_airport_df = spark.table(
    "capstone_3flight.flight_schema.airport"
)

# Calculate final row counts
airline_count = saved_airline_df.count()
airport_count = saved_airport_df.count()

print("========== FINAL ROW COUNTS ==========")
print(f"Airline ID Table : {airline_count}")
print(f"Airport ID Table : {airport_count}")
print(f"Total Records    : {airline_count + airport_count}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
from functools import reduce

CATALOG = "capstone_3flight"
SCHEMA = "flight_schema"
VOLUME = "ext_capstone3_vol"

BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
PRIMARY_PATH = f"{BASE_PATH}/primary_events/"
STATUS_PATH = f"{BASE_PATH}/status_updates/"
SECONDARY_PATH = f"{BASE_PATH}/secondary_events/"

AUTOLOADER_SCHEMA_PATH = f"{BASE_PATH}/autoloader_schema/primary_events"
PRIMARY_CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints/bronze_flight_events"

BRONZE_FLIGHT_TABLE = f"{CATALOG}.{SCHEMA}.bronze_flight_events"
BRONZE_CDC_TABLE = f"{CATALOG}.{SCHEMA}.bronze_status_cdc"
BRONZE_SECONDARY_TABLE = f"{CATALOG}.{SCHEMA}.bronze_secondary_events"

AIRLINE_TABLE = f"{CATALOG}.{SCHEMA}.airline"
AIRPORT_TABLE = f"{CATALOG}.{SCHEMA}.airport"

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print("Catalog:", CATALOG)
print("Schema:", SCHEMA)
print("Volume root:", BASE_PATH)

In [0]:
required_source_folders = {
    "primary_events": PRIMARY_PATH,
    "status_updates": STATUS_PATH,
    "secondary_events": SECONDARY_PATH,
}

for folder_name, folder_path in required_source_folders.items():
    try:
        entries = dbutils.fs.ls(folder_path)
        csv_files = [x for x in entries if x.name.lower().endswith(".csv")]
        print(f"{folder_name}: {len(csv_files)} CSV file(s)")
        for item in csv_files:
            print("   ", item.name, item.size, "bytes")
    except Exception as e:
        print(f"ERROR reading {folder_name} at {folder_path}: {e}")
        raise

dbutils.fs.mkdirs(AUTOLOADER_SCHEMA_PATH)
dbutils.fs.mkdirs(PRIMARY_CHECKPOINT_PATH)


In [0]:
if not RUN_PROFILING:
    print("Profiling skipped (run_profiling=false)")
if RUN_PROFILING:
    primary_file_infos = [
        x for x in dbutils.fs.ls(PRIMARY_PATH)
        if x.name.lower().endswith(".csv")
    ]
    primary_file_infos = sorted(primary_file_infos, key=lambda x: x.name)
    
    print("Primary CSV file count:", len(primary_file_infos))
    display(spark.createDataFrame(
        [(x.name, x.path, x.size) for x in primary_file_infos],
        ["file_name", "file_path", "size_bytes"]
    ))
    
    primary_dfs = []
    primary_file_counts = []
    
    for file_info in primary_file_infos:
        one_file_df = (
            spark.read
            .format("csv")
            .option("header", "true")
            .option("inferSchema", "true")
            .option("mode", "PERMISSIVE")
            .load(file_info.path)
            .withColumn("_source_file", F.lit(file_info.path))
        )
        primary_dfs.append(one_file_df)
        primary_file_counts.append(
            (file_info.name, one_file_df.count(), len(one_file_df.columns))
        )
    
    if not primary_dfs:
        raise ValueError(f"No CSV files found in {PRIMARY_PATH}")
    
    primary_raw = reduce(
        lambda left, right: left.unionByName(right, allowMissingColumns=True),
        primary_dfs
    )
    
    print("Primary total rows:", primary_raw.count())
    print("Combined columns:", len(primary_raw.columns))
    print("Combined column names:", primary_raw.columns)
    display(primary_raw.limit(10))

In [0]:
if not RUN_PROFILING:
    print("Profiling skipped (run_profiling=false)")
if RUN_PROFILING:
    file_count_df = spark.createDataFrame(
        primary_file_counts,
        ["file_name", "row_count", "column_count"]
    ).orderBy("file_name")
    
    display(file_count_df)
    
    print("Files with 13 columns (schema-evolved batches):")
    display(file_count_df.filter(F.col("column_count") == 13))
    
    print("Primary schema:")
    primary_raw.printSchema()

In [0]:
if not RUN_PROFILING:
    print("Profiling skipped (run_profiling=false)")
if RUN_PROFILING:
    schema_evolution_report = spark.createDataFrame(
        [
            (
                x.name,
                "source_schema_version" in next(
                    d.columns for d, info in zip(primary_dfs, primary_file_infos)
                    if info.name == x.name
                )
            )
            for x in primary_file_infos
        ],
        ["file_name", "has_source_schema_version"]
    ).orderBy("file_name")
    
    display(schema_evolution_report)
    
    if "source_schema_version" in primary_raw.columns:
        display(
            primary_raw
            .groupBy("_source_file", "source_schema_version")
            .count()
            .orderBy("_source_file", "source_schema_version")
        )
    else:
        print("WARNING: source_schema_version was not detected in the combined source data.")

In [0]:
if not RUN_PROFILING:
    print("Profiling skipped (run_profiling=false)")
if RUN_PROFILING:
    base_columns = [
        "flight_id", "event_timestamp", "ingest_timestamp",
        "airline_id", "airport_id", "flight_type", "flight_status",
        "timezone", "fare_amount", "delay_minutes", "aircraft_id",
        "source_system"
    ]
    
    missing_base_columns = [c for c in base_columns if c not in primary_raw.columns]
    if missing_base_columns:
        raise ValueError(f"Missing expected primary columns: {missing_base_columns}")
    
    primary_total_rows = primary_raw.count()
    primary_unique_flight_ids = primary_raw.select("flight_id").distinct().count()
    
    duplicate_flight_id_groups = (
        primary_raw.groupBy("flight_id")
        .count()
        .filter(F.col("count") > 1)
        .orderBy("flight_id")
    )
    
    exact_duplicate_groups = (
        primary_raw.groupBy(*base_columns)
        .count()
        .filter(F.col("count") > 1)
    )
    
    duplicate_extra_rows = (
        exact_duplicate_groups
        .select(F.sum(F.col("count") - F.lit(1)).alias("duplicate_extra_rows"))
        .first()["duplicate_extra_rows"]
    )
    
    print("Total primary rows:", primary_total_rows)
    print("Distinct flight IDs:", primary_unique_flight_ids)
    print("Flight IDs occurring more than once:", duplicate_flight_id_groups.count())
    print("Exact duplicate groups:", exact_duplicate_groups.count())
    print("Extra duplicate rows:", duplicate_extra_rows)
    
    display(duplicate_flight_id_groups)
    display(exact_duplicate_groups)

In [0]:
if not RUN_PROFILING:
    print("Profiling skipped (run_profiling=false)")
if RUN_PROFILING:
    null_profile = primary_raw.select([
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in primary_raw.columns
    ])
    display(null_profile)
    
    print(
        "flight_type null count:",
        primary_raw.filter(F.col("flight_type").isNull()).count()
    )
    
    if "source_schema_version" in primary_raw.columns:
        print(
            "source_schema_version null count (expected on older batches):",
            primary_raw.filter(F.col("source_schema_version").isNull()).count()
        )

In [0]:
if not RUN_PROFILING:
    print("Profiling skipped (run_profiling=false)")
if RUN_PROFILING:
    display(
        primary_raw
        .groupBy("flight_status")
        .count()
        .orderBy("flight_status")
    )

In [0]:
if not RUN_PROFILING:
    print("Profiling skipped (run_profiling=false)")
if RUN_PROFILING:
    primary_profile_typed = (
        primary_raw
        .withColumn("fare_amount_num", F.col("fare_amount").cast("double"))
        .withColumn("delay_minutes_num", F.col("delay_minutes").cast("double"))
    )
    
    display(
        primary_profile_typed
        .select(
            F.min("fare_amount_num").alias("min_fare"),
            F.max("fare_amount_num").alias("max_fare"),
            F.avg("fare_amount_num").alias("avg_fare"),
            F.min("delay_minutes_num").alias("min_delay"),
            F.max("delay_minutes_num").alias("max_delay"),
            F.avg("delay_minutes_num").alias("avg_delay")
        )
    )
    
    outlier_fares = (
        primary_profile_typed
        .filter(F.col("fare_amount_num") > 5000)
        .select("flight_id", "fare_amount", "delay_minutes", "_source_file")
        .orderBy(F.col("fare_amount_num").desc())
    )
    
    print("Fare values greater than 5000:", outlier_fares.count())
    display(outlier_fares)

In [0]:
if not RUN_PROFILING:
    print("Profiling skipped (run_profiling=false)")
if RUN_PROFILING:
    display(
        primary_raw
        .groupBy("timezone")
        .count()
        .orderBy(F.desc("count"))
    )
    
    display(
        primary_raw
        .select("event_timestamp", "ingest_timestamp")
        .orderBy("event_timestamp")
        .limit(10)
    )
    
    print("Distinct timezone values:", primary_raw.select("timezone").distinct().count())
    print(
        "Rows where timezone = USD:",
        primary_raw.filter(F.upper(F.col("timezone")) == "USD").count()
    )

In [0]:
if not RUN_PROFILING:
    print("Profiling skipped (run_profiling=false)")
if RUN_PROFILING:
    status_raw = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("mode", "PERMISSIVE")
        .load(STATUS_PATH)
    )
    
    print("CDC row count:", status_raw.count())
    print("CDC columns:", status_raw.columns)
    status_raw.printSchema()
    display(status_raw.limit(10))

In [0]:
if not RUN_PROFILING:
    print("Profiling skipped (run_profiling=false)")
if RUN_PROFILING:
    display(
        status_raw
        .groupBy("new_status")
        .count()
        .orderBy("new_status")
    )
    
    cdc_total = status_raw.count()
    cdc_distinct_sequences = status_raw.select("source_sequence").distinct().count()
    cdc_duplicate_sequences = (
        status_raw.groupBy("source_sequence")
        .count()
        .filter(F.col("count") > 1)
    )
    
    print("CDC rows:", cdc_total)
    print("Distinct source_sequence values:", cdc_distinct_sequences)
    print("Duplicate source_sequence groups:", cdc_duplicate_sequences.count())
    display(cdc_duplicate_sequences)
    

In [0]:
if not RUN_PROFILING:
    print("Profiling skipped (run_profiling=false)")
if RUN_PROFILING:
    secondary_raw = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("mode", "PERMISSIVE")
        .load(SECONDARY_PATH)
    )
    
    print("Secondary event rows:", secondary_raw.count())
    print("Secondary columns:", secondary_raw.columns)
    secondary_raw.printSchema()
    display(secondary_raw.limit(10))

In [0]:
if not RUN_PROFILING:
    print("Profiling skipped (run_profiling=false)")
if RUN_PROFILING:
    airline_df = spark.table(AIRLINE_TABLE)
    airport_df = spark.table(AIRPORT_TABLE)
    
    airline_count = airline_df.count()
    airport_count = airport_df.count()
    
    print("Airline table:", AIRLINE_TABLE, "| rows:", airline_count)
    print("Airport table:", AIRPORT_TABLE, "| rows:", airport_count)
    
    print("Airline schema:")
    airline_df.printSchema()
    print("Airport schema:")
    airport_df.printSchema()
    
    display(airline_df.limit(10))
    display(airport_df.limit(10))

In [0]:
if not RUN_PROFILING:
    print("Profiling skipped (run_profiling=false)")
if RUN_PROFILING:
    for table_name, df, key_col in [
        (AIRLINE_TABLE, airline_df, "airline_id"),
        (AIRPORT_TABLE, airport_df, "airport_id")
    ]:
        print(f"\nValidation for {table_name}")
        print("Rows:", df.count())
        print("Distinct keys:", df.select(key_col).distinct().count())
        print("Null keys:", df.filter(F.col(key_col).isNull()).count())
        print("Duplicate key groups:",
              df.groupBy(key_col).count().filter(F.col("count") > 1).count())

In [0]:
EXPECTED_PRIMARY_ROWS = 11540
EXPECTED_UNIQUE_FLIGHTS = 11500
EXPECTED_DUPLICATE_ROWS = 40
EXPECTED_FLIGHT_TYPE_NULLS = 12
EXPECTED_OUTLIER_FARES = 35
EXPECTED_CDC_ROWS = 520
EXPECTED_SECONDARY_ROWS = 310
EXPECTED_AIRLINE_ROWS = 110
EXPECTED_AIRPORT_ROWS = 150

In [0]:
def qa_status(actual, expected):
    return "PASS" if actual == expected else "CHECK"


# Source-profiling reconciliation needs the profiling cells above.
if not RUN_PROFILING:
    print("Profiling reconciliation skipped (run_profiling=false)")
if RUN_PROFILING:
    observed_checks = [
        ("Primary rows", primary_total_rows, EXPECTED_PRIMARY_ROWS),
        ("Distinct flight IDs", primary_unique_flight_ids, EXPECTED_UNIQUE_FLIGHTS),
        ("Extra exact duplicate rows", int(duplicate_extra_rows or 0), EXPECTED_DUPLICATE_ROWS),
        ("Null flight_type rows", primary_raw.filter(F.col("flight_type").isNull()).count(), EXPECTED_FLIGHT_TYPE_NULLS),
        ("Fare outliers > 5000", outlier_fares.count(), EXPECTED_OUTLIER_FARES),
        ("CDC rows", cdc_total, EXPECTED_CDC_ROWS),
        ("Secondary event rows", secondary_raw.count(), EXPECTED_SECONDARY_ROWS),
        ("Airline reference rows", airline_count, EXPECTED_AIRLINE_ROWS),
        ("Airport reference rows", airport_count, EXPECTED_AIRPORT_ROWS),
    ]
    
    reconciliation_df = spark.createDataFrame(
        [(name, int(actual), int(expected), qa_status(actual, expected))
         for name, actual, expected in observed_checks],
        ["check_name", "actual_value", "expected_value", "result"]
    )
    
    display(reconciliation_df)

In [0]:
# =====================================================
# BRONZE RUN HELPERS + PRE-INGESTION SAFETY CHECKS (read-only)
# =====================================================
import time
from urllib.parse import unquote

LINEAGE_COLS = ["_source_file", "_ingested_at"]
MIN_FILE_AGE_SECONDS = 300          # files modified more recently than this are deferred to the next run
BRONZE_TARGET_TABLES = [BRONZE_FLIGHT_TABLE, BRONZE_CDC_TABLE, BRONZE_SECONDARY_TABLE]
BRONZE_TABLES = BRONZE_TARGET_TABLES + [AIRLINE_TABLE, AIRPORT_TABLE]

# Path formats differ by origin (verified read-only on 2026-11-03):
#   dbutils.fs.ls()                                  -> dbfs:/Volumes/...
#   bronze_flight_events._source_file (Auto Loader)  -> /Volumes/...        (no dbfs: prefix)
#   bronze_status_cdc / bronze_secondary_events      -> dbfs:/Volumes/...   (batch _metadata.file_path)
# norm_path() maps every form to /Volumes/... and is used for EVERY path comparison below.
def norm_path(p):
    p = unquote(p)
    return p[len("dbfs:"):] if p.startswith("dbfs:") else p


def list_source_csvs(folder):
    """{normalised path: FileInfo} for the CSV files in a source folder."""
    out = {}
    for f in dbutils.fs.ls(folder):
        if f.name.lower().endswith(".csv"):
            k = norm_path(f.path)
            if k in out:
                raise RuntimeError(f"Two source paths normalise to {k}")
            out[k] = f
    return out


def table_file_stats(table):
    """{normalised _source_file: (rows, max _ingested_at in epoch ms)};
    fails on NULL/blank _source_file, NULL _ingested_at or colliding paths."""
    df = spark.table(table)
    bad_lineage = df.filter(F.col("_source_file").isNull() | (F.trim(F.col("_source_file")) == "")
                            | F.col("_ingested_at").isNull())
    if bad_lineage.limit(1).count():
        raise RuntimeError(f"{table} has rows with NULL/blank _source_file or NULL _ingested_at - lineage incomplete")
    stats = {}
    for r in (df.groupBy("_source_file")
                .agg(F.count("*").alias("n"), F.expr("unix_millis(max(_ingested_at))").alias("ms"))
                .collect()):
        k = norm_path(r["_source_file"])
        if k in stats:
            raise RuntimeError(f"{table}: two stored _source_file values normalise to {k}")
        stats[k] = (r["n"], r["ms"])
    return stats


def file_fingerprints(df, cols):
    """PRACTICAL per-file fingerprint {normalised file: (row_count, sum of xxhash64 over the typed columns)}.
    Order-independent and counts duplicate rows every time, but NOT an exact content comparison:
    hash collisions are possible in principle, and text that parses to the same typed value is not distinguished."""
    rows = (df.groupBy("_source_file")
              .agg(F.count("*").alias("n"),
                   F.sum(F.xxhash64(*[F.col(c) for c in cols]).cast("decimal(38,0)")).alias("h"))
              .collect())
    out = {}
    for r in rows:
        k = norm_path(r["_source_file"])
        if k in out:
            raise RuntimeError(f"Two _source_file values normalise to {k}")
        out[k] = (r["n"], r["h"])
    return out


def read_bronze_csv(paths, src_schema):
    """Read source CSVs with the EXISTING Bronze table schema (no inference, no schema change)."""
    return (spark.read.format("csv")
            .option("header", "true")
            .option("enforceSchema", "false")     # header must match the existing table columns
            .option("mode", "FAILFAST")           # never silently null a raw value
            .schema(src_schema)
            .load(paths)
            .withColumn("_source_file", F.col("_metadata.file_path")))


def _dir_entries(path):
    try:
        return {f.name.rstrip("/") for f in dbutils.fs.ls(path)}
    except Exception:
        return set()


# ---- Safety check 1: the production Bronze tables must already exist (never created or replaced here) ----
missing_tables = [t for t in BRONZE_TABLES if not spark.catalog.tableExists(t)]
if missing_tables:
    raise RuntimeError(f"Production Bronze table(s) missing: {missing_tables}. "
                       "Refusing to continue - this notebook never creates or replaces production tables.")

# ---- Safety check 2: existing Auto Loader state must be present when the target already holds data ----
# (a missing or empty checkpoint would make Auto Loader re-ingest every primary file)
if spark.table(BRONZE_FLIGHT_TABLE).limit(1).count() > 0:
    checkpoint_entries = _dir_entries(PRIMARY_CHECKPOINT_PATH)
    schema_entries = _dir_entries(AUTOLOADER_SCHEMA_PATH)
    if not {"offsets", "commits"}.issubset(checkpoint_entries):
        raise RuntimeError(f"Auto Loader checkpoint {PRIMARY_CHECKPOINT_PATH} is missing or incomplete "
                           f"({sorted(checkpoint_entries)}) while {BRONZE_FLIGHT_TABLE} has data - stopping to avoid re-ingestion")
    if "_schemas" not in schema_entries:
        raise RuntimeError(f"Auto Loader schema location {AUTOLOADER_SCHEMA_PATH} has no _schemas folder "
                           f"({sorted(schema_entries)}) while {BRONZE_FLIGHT_TABLE} has data - stopping")

# ---- Safety check 3: lineage columns must exist, be complete and be plausible in every Bronze target table ----
lineage_problems = {}
for t in BRONZE_TARGET_TABLES:
    missing_lineage = [c for c in LINEAGE_COLS if c not in spark.table(t).columns]
    if missing_lineage:
        lineage_problems[t] = {"missing_columns": missing_lineage}
        continue
    lin = (spark.table(t).agg(
        F.count("*").alias("rows"),
        F.sum(F.col("_source_file").isNull().cast("int")).alias("null_source_file"),
        F.sum((F.trim(F.col("_source_file")) == "").cast("int")).alias("blank_source_file"),
        F.sum(F.col("_ingested_at").isNull().cast("int")).alias("null_ingested_at"),
        F.sum((F.col("_ingested_at") > F.current_timestamp() + F.expr("INTERVAL 1 HOUR")).cast("int"))
         .alias("future_ingested_at"))
        .first())
    issues = {k: lin[k] for k in ["null_source_file", "blank_source_file", "null_ingested_at", "future_ingested_at"]
              if (lin[k] or 0) > 0}
    print(f"Lineage check {t.split('.')[-1]}: rows {lin['rows']} | issues {issues or 'none'}")
    if issues:
        lineage_problems[t] = issues
if lineage_problems:
    raise RuntimeError(f"Bronze lineage columns missing or invalid - stopping before ingestion: {lineage_problems}")

# ---- Pre-ingestion snapshot ----
BEFORE_COUNTS = {t: spark.table(t).count() for t in BRONZE_TABLES}
PRIMARY_FILES_BEFORE = {k: v[0] for k, v in table_file_stats(BRONZE_FLIGHT_TABLE).items()}
INGESTION_START_MS = int(time.time() * 1000)

print("Production Bronze tables present:", [t.split(".")[-1] for t in BRONZE_TABLES])
print("Auto Loader checkpoint:", PRIMARY_CHECKPOINT_PATH)
print("Auto Loader schema location:", AUTOLOADER_SCHEMA_PATH)
print("Row counts before ingestion:", {t.split(".")[-1]: n for t, n in BEFORE_COUNTS.items()})
print("Primary files already in Bronze:", len(PRIMARY_FILES_BEFORE))

In [0]:
primary_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.schemaLocation", AUTOLOADER_SCHEMA_PATH)
    .option("rescuedDataColumn", "_rescued_data")
    .load(PRIMARY_PATH)
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

primary_bronze_query = (
    primary_stream.writeStream
    .format("delta")
    .option("checkpointLocation", PRIMARY_CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .toTable(BRONZE_FLIGHT_TABLE)
)

primary_bronze_query.awaitTermination()

print("Auto Loader primary ingestion completed.")
print("Bronze table:", BRONZE_FLIGHT_TABLE)

In [0]:
# =====================================================
# INITIAL AUTO LOADER INGESTION VALIDATION (read-only)
# Note: bronze_flight_events stores _source_file as /Volumes/... while dbutils.fs.ls returns
# dbfs:/Volumes/... - every comparison below goes through norm_path().
#
# PRIMARY FILE IMMUTABILITY REQUIREMENT
#   Primary source files must be treated as immutable once landed. Auto Loader does not re-ingest a file that
#   is replaced under the same name, and this cell does NOT verify primary file contents. It detects only:
#     - a changed per-file row count (FAIL), and
#     - a modification time newer than the file's ingestion time (WARNING).
#   A replacement with the same row count but different values may go undetected.
#   A per-file content fingerprint (as used for CDC / secondary) was evaluated and NOT adopted for primary files:
#     1. The Auto Loader schema has evolved (source_schema_version exists only in newer files), so one fixed-schema
#        batch re-read cannot match every file version by header name.
#     2. Auto Loader parses with inferred column types and _rescued_data semantics (mismatching values are rescued and
#        the column nulled). A batch re-read does not reproduce that exactly, so unmodified files could produce
#        different hashes and fail falsely.
#     3. Storing an ingestion-time fingerprint would need new columns or a new manifest table, which is out of scope.
# =====================================================
AUTOLOADER_PROGRESS_ROWS = sum(
    (p.get("numInputRows") or 0)
    for p in primary_bronze_query.recentProgress
)
primary_after = table_file_stats(BRONZE_FLIGHT_TABLE)
primary_source = list_source_csvs(PRIMARY_PATH)
al_problems, al_warnings = [], []

# 1. files that were already ingested must not change
changed_old = [k for k, n in PRIMARY_FILES_BEFORE.items() if primary_after.get(k, (None,))[0] != n]
if changed_old:
    al_problems.append(f"previously ingested files changed row count: {changed_old}")

# 2. rows added must come only from files that are new in this run
PRIMARY_NEW_FILES = sorted(set(primary_after) - set(PRIMARY_FILES_BEFORE))
PRIMARY_ROWS_ADDED = spark.table(BRONZE_FLIGHT_TABLE).count() - BEFORE_COUNTS[BRONZE_FLIGHT_TABLE]
PRIMARY_ROWS_IN_NEW_FILES = sum(primary_after[k][0] for k in PRIMARY_NEW_FILES)
if PRIMARY_ROWS_ADDED != PRIMARY_ROWS_IN_NEW_FILES:
    al_problems.append(f"rows added {PRIMARY_ROWS_ADDED} != rows in newly ingested files {PRIMARY_ROWS_IN_NEW_FILES}")

# 3. every source file present before this run started must now be in Bronze (header-only files excepted)
not_in_bronze = [k for k in primary_source if k not in primary_after]
late_files = [k for k in not_in_bronze if primary_source[k].modificationTime >= INGESTION_START_MS]
missed_files = [k for k in not_in_bronze if k not in late_files]
non_empty_missed = [k for k in missed_files
                    if spark.read.option("header", "true").csv(primary_source[k].path).limit(1).count() > 0]
if non_empty_missed:
    al_problems.append(f"source files not ingested by Auto Loader: {non_empty_missed}")
if late_files:
    al_warnings.append(f"files arrived during the run (ingested by the replay or the next run): {late_files}")

# 4. stored paths no longer in the folder, and files modified after ingestion (Auto Loader does not re-ingest them)
gone_files = sorted(set(primary_after) - set(primary_source))
if gone_files:
    al_warnings.append(f"Bronze rows reference files no longer in the folder: {gone_files}")
modified_files = [k for k in primary_after
                  if k in primary_source and primary_source[k].modificationTime > primary_after[k][1]]
if modified_files:
    al_warnings.append(f"PRIMARY FILE MODIFIED AFTER INGESTION - Auto Loader will NOT re-ingest it: {modified_files}")

# 5. per-file row count: current source file vs Bronze (detects replacements that change the row count).
#    Untyped CSV read (no schema), so it does not depend on, or interfere with, the Auto Loader schema or _rescued_data.
present_files = sorted(k for k in primary_after if k in primary_source)
if present_files:
    source_row_counts = {
        norm_path(r["f"]): r["n"]
        for r in (spark.read.format("csv").option("header", "true")
                  .load([primary_source[k].path for k in present_files])
                  .select(F.col("_metadata.file_path").alias("f"))
                  .groupBy("f").agg(F.count("*").alias("n"))
                  .collect())}
    count_mismatch = {k: {"source_rows": source_row_counts.get(k, 0), "bronze_rows": primary_after[k][0]}
                      for k in present_files if source_row_counts.get(k, 0) != primary_after[k][0]}
    if count_mismatch:
        al_problems.append(f"PRIMARY SOURCE FILE ROW COUNT CHANGED SINCE INGESTION (file replaced; Auto Loader will "
                           f"not re-ingest it): {count_mismatch}")
print(f"Primary per-file row-count check: {len(present_files)} file(s) compared (contents not verified - see note above)")

if AUTOLOADER_PROGRESS_ROWS != PRIMARY_ROWS_ADDED:
    al_warnings.append(f"Auto Loader progress reported {AUTOLOADER_PROGRESS_ROWS} rows, table grew by {PRIMARY_ROWS_ADDED} "
                       "(expected only after a retried micro-batch)")

print(f"New primary files this run: {len(PRIMARY_NEW_FILES)} | rows added: {PRIMARY_ROWS_ADDED}")
for w in al_warnings:
    print("WARNING |", w)
for p in al_problems:
    print("FAIL    |", p)
assert not al_problems, "Initial Auto Loader ingestion validation failed"
print("PASS: initial Auto Loader ingestion validated.")

In [0]:
bronze_flights = spark.table(BRONZE_FLIGHT_TABLE)

bronze_primary_count = bronze_flights.count()
bronze_distinct_flights = bronze_flights.select("flight_id").distinct().count()

print("Bronze flight rows:", bronze_primary_count)
print("Distinct flight IDs:", bronze_distinct_flights)
print("Columns:", bronze_flights.columns)
bronze_flights.printSchema()

display(
    bronze_flights
    .groupBy("_source_file")
    .count()
    .orderBy("_source_file")
)

display(
    bronze_flights
    .select("_source_file", "_ingested_at")
    .limit(10)
)

In [0]:
print(
    "Bronze rows retained beyond distinct flight IDs:",
    bronze_primary_count - bronze_distinct_flights
)

print(
    "source_schema_version present:",
    "source_schema_version" in bronze_flights.columns
)

if "source_schema_version" in bronze_flights.columns:
    display(
        bronze_flights
        .groupBy("source_schema_version")
        .count()
        .orderBy("source_schema_version")
    )

In [0]:

# =====================================================
# INCREMENTAL CDC + SECONDARY INGESTION
# Appends only source files not yet in Bronze (by normalised _source_file), one atomic commit per table.
# No overwrite, no schema change, no table creation. Duplicate rows inside a file are kept as raw records.
#
# File verification uses a PRACTICAL FINGERPRINT per file: (row count, sum of xxhash64 over the typed source
# columns). It is order-independent and counts duplicate rows every time, but it is NOT an exact content
# comparison: hash collisions are possible in principle, and text that parses to the same typed value
# (for example "1.0" vs "1") is not distinguished.
# =====================================================
def load_new_csv_files_to_bronze(source_path, target_table):
    if not spark.catalog.tableExists(target_table):
        raise RuntimeError(f"{target_table} not found - refusing to create it")
    tgt_schema = spark.table(target_table).schema
    missing_lineage = [c for c in LINEAGE_COLS if c not in tgt_schema.fieldNames()]
    if missing_lineage:
        raise RuntimeError(f"{target_table}: lineage column(s) {missing_lineage} missing - nothing written")
    src_cols = [f.name for f in tgt_schema if f.name not in LINEAGE_COLS]
    src_schema = T.StructType([f for f in tgt_schema if f.name not in LINEAGE_COLS])

    before = table_file_stats(target_table)
    source = list_source_csvs(source_path)

    # PRE-FLIGHT 1: every stored path must match a file in the folder (otherwise path matching is broken -> stop)
    orphan = sorted(set(before) - set(source))
    if orphan:
        raise RuntimeError(f"{target_table}: stored _source_file not found in {source_path}: {orphan} - nothing written")

    # PRE-FLIGHT 2: already-ingested files must still match what was ingested (row count + practical fingerprint)
    tbl_fp_before = file_fingerprints(spark.table(target_table), src_cols)
    if before:
        src_fp = file_fingerprints(read_bronze_csv([source[k].path for k in sorted(before)], src_schema), src_cols)
        changed = sorted(k for k in before if src_fp.get(k) != tbl_fp_before.get(k))
        if changed:
            raise RuntimeError(f"{target_table}: MODIFIED_OR_INCONSISTENT_SOURCE_FILE {changed} - source content no longer "
                               "matches Bronze (file replaced under the same name, or loaded more than once); nothing written")
    touched = [k for k in before if source[k].modificationTime > before[k][1]]
    if touched:
        print(f"WARNING | {target_table}: files modified after ingestion but content unchanged: {touched}")

    now_ms = int(time.time() * 1000)
    deferred = sorted(k for k in source if k not in before
                      and now_ms - source[k].modificationTime < MIN_FILE_AGE_SECONDS * 1000)
    new_keys = sorted(k for k in source if k not in before and k not in deferred)

    # PRE-APPEND 1: header compatibility - every new file must carry exactly the existing column names, in order
    skipped, header_problems = {}, {}
    for k in new_keys:
        head = (spark.read.format("csv").option("header", "false")
                .load(source[k].path).limit(2).collect())
        if not head:
            skipped[k] = "empty file (no header, no data rows)"
            continue
        header = [None if c is None else str(c).lstrip("\ufeff").strip() for c in head[0]]
        if header != src_cols:
            header_problems[k] = f"header {header} != existing columns {src_cols}"
        elif len(head) == 1:
            skipped[k] = "header-only file (0 data rows)"
    if header_problems:
        raise RuntimeError(f"{target_table}: incompatible source header(s) - nothing written: {header_problems}")
    to_load = [k for k in new_keys if k not in skipped]

    # PRE-APPEND 2: schema compatibility - parse every new file with the EXISTING column types (FAILFAST,
    # header checked again by name). The fingerprint is a full read, so any value that does not fit an existing
    # type fails HERE, before the append. Duplicate rows are kept as-is.
    new_fp = {}
    if to_load:
        new_src = read_bronze_csv([source[k].path for k in to_load], src_schema)
        new_fp = file_fingerprints(new_src, src_cols)          # what is about to be written, per file
        no_rows = [k for k in to_load if k not in new_fp]
        if no_rows:
            raise RuntimeError(f"{target_table}: file(s) produced no rows on the typed read: {no_rows} - nothing written")
        (new_src.withColumn("_ingested_at", F.current_timestamp())
                .select(*tgt_schema.fieldNames())
                .write.format("delta").mode("append").saveAsTable(target_table))   # one atomic commit

    # POST-WRITE: file-level verification
    tbl_fp_after = file_fingerprints(spark.table(target_table), src_cols)
    problems = []
    for k in before:                                            # previously ingested files untouched
        if tbl_fp_after.get(k) != tbl_fp_before.get(k):
            problems.append(f"previously ingested file changed: {k}")
    for k in to_load:                                           # each loaded file present once, same count + fingerprint
        if tbl_fp_after.get(k) != new_fp.get(k):
            problems.append(f"new file not ingested exactly once: {k} expected {new_fp.get(k)} got {tbl_fp_after.get(k)}")
    for k in skipped:                                           # skipped files must not appear in Bronze
        if k in tbl_fp_after:
            problems.append(f"skipped file unexpectedly present in Bronze: {k}")
    unexpected = sorted(set(tbl_fp_after) - set(before) - set(to_load))
    if unexpected:
        problems.append(f"files appeared that this run did not load (concurrent run?): {unexpected}")
    rows_added = sum(v[0] for v in tbl_fp_after.values()) - sum(v[0] for v in tbl_fp_before.values())
    expected_added = sum(v[0] for v in new_fp.values())
    if rows_added != expected_added:
        problems.append(f"rows added {rows_added} != rows read from new files {expected_added}")
    if problems:
        raise RuntimeError(f"{target_table}: file-level verification failed: {problems}")

    print(f"{target_table}: files {len(source)} | already loaded {len(before)} | ingested now {len(to_load)} "
          f"| rows added {rows_added} | skipped {len(skipped)} | deferred {len(deferred)}")
    for k in to_load:
        print(f"INGESTED | {target_table} | {k} | rows {new_fp[k][0]}")
    for k, reason in sorted(skipped.items()):
        print(f"SKIPPED  | {target_table} | {k} | {reason} - NOT ingested; re-checked on every run")
    for k in deferred:
        print(f"DEFERRED | {target_table} | {k} | modified < {MIN_FILE_AGE_SECONDS}s ago - next run")
    return {"files_loaded": len(to_load), "rows_added": rows_added, "deferred": deferred,
            "skipped": sorted(skipped)}


LOAD_RESULTS, LOAD_ERRORS = {}, {}
for src_path, tbl in [(STATUS_PATH, BRONZE_CDC_TABLE), (SECONDARY_PATH, BRONZE_SECONDARY_TABLE)]:
    try:
        LOAD_RESULTS[tbl] = load_new_csv_files_to_bronze(src_path, tbl)
    except Exception as e:                       # keep going so the other table can still load
        LOAD_ERRORS[tbl] = f"{type(e).__name__}: {e}"
        print(f"FAILED | {tbl} | {LOAD_ERRORS[tbl]}")

bronze_cdc = spark.table(BRONZE_CDC_TABLE)
bronze_secondary = spark.table(BRONZE_SECONDARY_TABLE)
if LOAD_ERRORS:
    raise RuntimeError(f"Incremental Bronze load failed for {list(LOAD_ERRORS)} - committed tables are safe; "
                       "a retry loads only the files still missing")


In [0]:
# =====================================================
# BRONZE COUNT RECONCILIATION
# dynamic check (always): rows added == rows actually ingested this run; reference tables unchanged
# strict check (strict_expected_counts=true): final counts == original capstone counts
# =====================================================
AFTER_COUNTS = {t: spark.table(t).count() for t in BRONZE_TABLES}
EXPECTED = {BRONZE_FLIGHT_TABLE: EXPECTED_PRIMARY_ROWS, BRONZE_CDC_TABLE: EXPECTED_CDC_ROWS,
            BRONZE_SECONDARY_TABLE: EXPECTED_SECONDARY_ROWS, AIRLINE_TABLE: EXPECTED_AIRLINE_ROWS,
            AIRPORT_TABLE: EXPECTED_AIRPORT_ROWS}
EXPECTED_ADDED = {BRONZE_FLIGHT_TABLE: PRIMARY_ROWS_IN_NEW_FILES,
                  BRONZE_CDC_TABLE: LOAD_RESULTS[BRONZE_CDC_TABLE]["rows_added"],
                  BRONZE_SECONDARY_TABLE: LOAD_RESULTS[BRONZE_SECONDARY_TABLE]["rows_added"],
                  AIRLINE_TABLE: 0, AIRPORT_TABLE: 0}          # reference tables are never written here

bronze_validation_rows = []
for t in BRONZE_TABLES:
    b, a = BEFORE_COUNTS[t], AFTER_COUNTS[t]
    dynamic_result = "PASS" if a - b == EXPECTED_ADDED[t] else "FAIL"
    strict_result = ("PASS" if a == EXPECTED[t] else "FAIL") if STRICT_EXPECTED_COUNTS else "SKIPPED"
    bronze_validation_rows.append((t.split(".")[-1], b, a, a - b, EXPECTED_ADDED[t], EXPECTED[t],
                                   dynamic_result, strict_result))

bronze_validation_df = spark.createDataFrame(
    bronze_validation_rows,
    "table_name string, rows_before long, rows_after long, rows_added long, new_rows_ingested long, "
    "capstone_expected long, dynamic_result string, strict_result string")
display(bronze_validation_df)

failed_tables = [r[0] for r in bronze_validation_rows if "FAIL" in (r[6], r[7])]
assert not failed_tables, f"Bronze count reconciliation failed for {failed_tables}"
print("PASS: Bronze count reconciliation", "(strict + dynamic)" if STRICT_EXPECTED_COUNTS else "(dynamic)")

In [0]:
display(
    bronze_cdc.select(
        "_source_file",
        "_ingested_at"
    ).distinct()
)

In [0]:
display(
    bronze_secondary.select(
        "_source_file",
        "_ingested_at"
    ).distinct()
)

In [0]:
# Informational summary (does not fail the run). Profiling row only when profiling ran.
day1_summary = []
if RUN_PROFILING:
    day1_summary.append(("Primary profiling", primary_total_rows, EXPECTED_PRIMARY_ROWS))
day1_summary += [
    ("Primary Bronze", AFTER_COUNTS[BRONZE_FLIGHT_TABLE], EXPECTED_PRIMARY_ROWS),
    ("CDC Bronze", AFTER_COUNTS[BRONZE_CDC_TABLE], EXPECTED_CDC_ROWS),
    ("Secondary Bronze", AFTER_COUNTS[BRONZE_SECONDARY_TABLE], EXPECTED_SECONDARY_ROWS),
    ("Airline reference", AFTER_COUNTS[AIRLINE_TABLE], EXPECTED_AIRLINE_ROWS),
    ("Airport reference", AFTER_COUNTS[AIRPORT_TABLE], EXPECTED_AIRPORT_ROWS),
]

display(spark.createDataFrame(
    [(name, int(actual), int(expected), qa_status(actual, expected))
     for name, actual, expected in day1_summary],
    ["component", "actual_rows", "expected_rows", "result"]
))

In [0]:
%sql
DESCRIBE TABLE capstone_3flight.flight_schema.bronze_flight_events;

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(_source_file) AS source_file_count,
    COUNT(_ingested_at) AS ingestion_timestamp_count
FROM capstone_3flight.flight_schema.bronze_flight_events;

In [0]:
bronze_table = "capstone_3flight.flight_schema.bronze_flight_events"

bronze_df = spark.table(bronze_table)

print("Does source_schema_version exist?",
      "source_schema_version" in bronze_df.columns)

print("Total columns:", len(bronze_df.columns))

display(bronze_df.printSchema())

In [0]:
from pyspark.sql import functions as F

display(
    bronze_df
    .withColumn(
        "batch_file",
        F.regexp_extract(
            F.col("_source_file"),
            r"(flight_batch_\d+\.csv)",
            1
        )
    )
    .groupBy("batch_file", "source_schema_version")
    .count()
    .orderBy("batch_file")
)

In [0]:
# =====================================================
# AUTO LOADER REPLAY (independent of the initial-ingestion validation)
# Re-runs the SAME streaming DataFrame defined in the Auto Loader cell (same source path, schema location,
# schema-evolution mode) with the SAME checkpoint, trigger and target table. Already processed files must add 0 rows.
# =====================================================
REPLAY_BEFORE_TOTAL = spark.table(BRONZE_FLIGHT_TABLE).count()
REPLAY_BEFORE_FILES = {k: v[0] for k, v in table_file_stats(BRONZE_FLIGHT_TABLE).items()}
print("Before replay:", REPLAY_BEFORE_TOTAL, "rows in", len(REPLAY_BEFORE_FILES), "files")

replay_query = (
    primary_stream.writeStream
    .format("delta")
    .option("checkpointLocation", PRIMARY_CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .toTable(BRONZE_FLIGHT_TABLE)
)
replay_query.awaitTermination()
REPLAY_PROGRESS_ROWS = sum(
    (p.get("numInputRows") or 0)
    for p in replay_query.recentProgress
)

In [0]:
replay_after_files = {k: v[0] for k, v in table_file_stats(BRONZE_FLIGHT_TABLE).items()}
replay_after_total = spark.table(BRONZE_FLIGHT_TABLE).count()
replay_rows_added = replay_after_total - REPLAY_BEFORE_TOTAL
reingested = [k for k, n in REPLAY_BEFORE_FILES.items() if replay_after_files.get(k) != n]
arrived_during_run = sorted(set(replay_after_files) - set(REPLAY_BEFORE_FILES))
rows_in_arrived = sum(replay_after_files[k] for k in arrived_during_run)

print("Before replay:", REPLAY_BEFORE_TOTAL)
print("After replay:", replay_after_total)
print("Replay progress rows:", REPLAY_PROGRESS_ROWS, "| rows added by replay:", replay_rows_added)

assert not reingested, f"Replay re-ingested already processed files: {reingested}"
assert replay_rows_added == rows_in_arrived, \
    f"Replay added {replay_rows_added} rows not explained by newly arrived files ({rows_in_arrived})"
if arrived_during_run:
    print(f"INFO: {len(arrived_during_run)} file(s) landed during the run and were ingested by the replay: {arrived_during_run}")
else:
    assert replay_rows_added == 0, f"Replay added {replay_rows_added} rows"

print("PASS: Auto Loader replay safety verified (existing checkpoint, zero re-ingested records).")

In [0]:
# =====================================================
# RUN SUMMARY + TASK VALUES (read-only; task values are ignored outside a job run)
# =====================================================
import json

FINAL_COUNTS = {t.split(".")[-1]: spark.table(t).count() for t in BRONZE_TABLES}
latest_primary_ingest = spark.table(BRONZE_FLIGHT_TABLE).agg(F.max("_ingested_at")).first()[0]

bronze_run_summary = {
    "bronze_status": "SUCCEEDED",
    "strict_expected_counts": STRICT_EXPECTED_COUNTS,
    "run_profiling": RUN_PROFILING,
    "bronze_final_counts": FINAL_COUNTS,
    "primary_new_files": len(PRIMARY_NEW_FILES),
    "primary_rows_added": PRIMARY_ROWS_ADDED,
    "cdc_rows_added": LOAD_RESULTS[BRONZE_CDC_TABLE]["rows_added"],
    "secondary_rows_added": LOAD_RESULTS[BRONZE_SECONDARY_TABLE]["rows_added"],
    "replay_rows_added": replay_rows_added,
    "latest_primary_ingested_at": str(latest_primary_ingest),
}
print(json.dumps(bronze_run_summary, indent=2, default=str))

try:
    for key, value in bronze_run_summary.items():
        dbutils.jobs.taskValues.set(key=key, value=value)
    print("Task values published.")
except Exception as e:
    print(f"Task values not published ({type(e).__name__}) - expected in interactive runs.")